# Data Warehousing

In [ ]:
import math
import numpy as np
import pandas as pd

import psycopg2


In [ ]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)
    

In [ ]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [ ]:
cursor = connection.cursor()

# Lab: Querying Dimensional Model - Single Star Schema

![Star Schema](star_schema.JPG)

##  Drop, create, and load the tables for the line item star schema

In [ ]:
connection.rollback()

query = """

drop table if exists line_item_facts;

drop table if exists receipt_dimension;

drop table if exists date_dimension;

drop table if exists customer_dimension;

drop table if exists store_dimension;

drop table if exists product_dimension;



"""

cursor.execute(query)

connection.commit()


In [ ]:
connection.rollback()

query = """

create table receipt_dimension (
  receipt_key numeric(12),
  receipt varchar(32),
  primary key (receipt_key)
);

create table date_dimension (
  date_key numeric(12),
  date_value date,
  dow numeric(1),
  dow_string varchar(9),
  month numeric(2),
  month_string varchar(9),
  primary key (date_key)
);

create table customer_dimension (
  customer_key numeric(12),
  customer_id numeric(6),
  first_name varchar(32),
  last_name varchar(32),
  street varchar(32),
  city varchar(32),
  state varchar(2),
  zip varchar(5),
  distance numeric(3),
  primary key (customer_key)
);


create table store_dimension (
  store_key numeric(12),
  store_id numeric(6),
  street varchar(32),
  city varchar(32),
  state varchar(2),
  zip varchar(5),
  latitude numeric(7,4),
  longitude numeric(7,4),
  primary key (store_key)
);


create table product_dimension (
  product_key numeric(12),
  product_id numeric(3),
  product_name varchar(32),
  primary key (product_key)  
);


create table line_item_facts (
  receipt_key numeric(12),
  date_key numeric(12),
  customer_key numeric(12),
  store_key numeric(12),
  product_key numeric(12),
  quantity numeric(3),
  price numeric(5,2),
  line_item_sub_total numeric(6),
  line_item_tax numeric(6),
  line_item_total numeric(6),
  primary key (receipt_key, date_key, customer_key, store_key, product_key),
  foreign key (receipt_key) references receipt_dimension (receipt_key),
  foreign key (date_key) references date_dimension (date_key),
  foreign key (customer_key) references customer_dimension (customer_key),
  foreign key (store_key) references store_dimension (store_key),
  foreign key (product_key) references product_dimension (product_key)
);


"""

cursor.execute(query)

connection.commit()

In [ ]:
connection.rollback()
    
query = """
    

copy receipt_dimension
from '/user/labs/week_14/receipt_dimension.csv' delimiter ',' NULL '' csv header;

copy date_dimension
from '/user/labs/week_14/date_dimension.csv' delimiter ',' NULL '' csv header;

copy customer_dimension
from '/user/labs/week_14/customer_dimension.csv' delimiter ',' NULL '' csv header;

copy store_dimension
from '/user/labs/week_14/store_dimension.csv' delimiter ',' NULL '' csv header;

copy product_dimension
from '/user/labs/week_14/product_dimension.csv' delimiter ',' NULL '' csv header;

copy line_item_facts
from  '/user/labs/week_14/line_item_facts.csv' delimiter ',' NULL '' csv header;

"""

cursor.execute(query)
    
connection.commit()


##  Queries to a star schema always use the same pattern: join the dimensions to the fact table;  use an inner join because "holes" and nulls are not allowed in star shemas

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select l.quantity,
       l.price,
       l.line_item_sub_total,
       l.line_item_tax,
       l.line_item_total,
       r.receipt_key,
       r.receipt,
       d.date_key,
       d.dow,
       d.dow_string,
       d.month,
       d.month_string,
       c.customer_key,
       c.customer_id,
       c.last_name,
       c.first_name,
       c.street,
       c.city,
       c.state,
       c.zip,
       c.distance,
       s.store_key,
       s.store_id,
       s.street as store_street,
       s.city as store_city,
       s.state as store_state,
       s.zip as store_zip,
       s.latitude,
       s.longitude,
       p.product_key,
       p.product_id,
       p.product_name
from line_item_facts as l
     join receipt_dimension as r
         on l.receipt_key = r.receipt_key
     join date_dimension as d
         on l.date_key = d.date_key
     join customer_dimension as c
         on l.customer_key = c.customer_key
     join store_dimension as s
         on l.store_key = s.store_key
     join product_dimension as p
         on l.product_key = p.product_key
order by r.receipt

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

##  You try it - using the star schema, for each receipt, find the subtotal, tax, and total amounts

# Lab: Querying Dimensional Model - Drilling Across Multiple Star Schemas

![Drilling Across](drilling_across.JPG)

##  Drop, create, and populate the tables that hold the results of star schema queries on the orders start schema and  the fulfillment star schema

In [ ]:
connection.rollback()

query = """

drop table if exists orders;

drop table if exists fulfillment;


"""

cursor.execute(query)

connection.commit()


In [ ]:
connection.rollback()

query = """

create table orders (
  order_id numeric(12),
  order_date date,
  sub_total numeric(5),
  tax numeric(5),
  total numeric(5)
);

create table fulfillment (
  fulfillment_id numeric(12),
  fulfillment_date date,
  order_id numeric(12),
  additional_order_id_1 numeric(12),
  additional_order_id_2 numeric(12),
  additional_order_id_3 numeric(12)
);

"""

cursor.execute(query)

connection.commit()

In [ ]:
connection.rollback()

query = """

insert into orders values(1, '2020-11-01', 36, 0, 36);
insert into orders values(2, '2020-11-02', 48, 0, 48);
insert into orders values(3, '2020-11-03', 24, 0, 24);
insert into orders values(4, '2020-11-04', 12, 0, 12);
insert into orders values(5, '2020-11-05', 36, 0, 36);
insert into orders values(6, '2020-11-06', 48, 0, 48);

insert into fulfillment values(11, '2020-11-05', 1, 0, 0, 0);
insert into fulfillment values(12, '2020-11-06', 1, 0, 0, 0);
insert into fulfillment values(13, '2020-11-07', 2, 3, 0, 0);
insert into fulfillment values(14, '2020-11-08', 4, 0, 0, 0);
insert into fulfillment values(15, '2020-11-09', 4, 0, 0, 0);
insert into fulfillment values(16, '2020-11-10', 5, 6, 0, 0);

"""

cursor.execute(query)

connection.commit()

##  The star schema orders query results

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from orders

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

##  The star schema fulfillment query results

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from fulfillment

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

##  Since orders and fulfillment are both star schemas, there is no defined relationship between them;  Essentially we need to do a "dangerous join" between the two fact tables, which can result in the "extra rows" problem and the "missing rows" problem

##  Let's do a dangerous join between orders and fulfillment and look at order 1;  order 1 was fulfilled (delivered) partially on two different days;  this create "extra rows" when we join them

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select o.order_id,
       o.order_date,
       o.sub_total,
       o.tax,
       o.total,
       f.fulfillment_id,
       f.fulfillment_date
from orders as o
     join fulfillment as f
         on o.order_id = f.order_id
where o.order_id = 1
order by 1

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

##  If we are not careful, we can double (or multi) count the extra rows, as in an aggregation; the aggregation below has double the sub total, tax, and total

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select o.order_id,
       o.order_date,
       sum(o.sub_total) as sub_total,
       sum(o.tax) as tax,
       sum(o.total) as total
from orders as o
     join fulfillment as f
         on o.order_id = f.order_id
where o.order_id = 1
group by o.order_id, o.order_date
order by 1

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

##  Fulfillment 13 delivered two orders to the customer on the same day, order 2 and order 3;  star schemas do not allow the fact table to be a parent of the dimension, so we have to denormalize the orders in the fulfillment record;  in our example, order 3 does not have a primary fulfillment record, which leads to the missing row problem

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select o.order_id,
       o.order_date,
       o.sub_total,
       o.tax,
       o.total,
       f.fulfillment_id,
       f.fulfillment_date
from orders as o
     join fulfillment as f
         on o.order_id = f.order_id
where o.order_id = 3
order by 1

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

##  "Drilling Across" is the solution;  drilling across means that we execute an independent query for each star schema, then write procedural code in a language such as Python to combine them with logic to handle the missing rows and extra rows problem; we cannot write SQL to handle a join on start schemas, it will never work!

##  You try it - find another extra rows problem and another missing rows problem when joining the orders star schema and the fulfillment star schema